# 11장. 벡터스토어와 데이터베이스 비교

이 장은 PDF 교재의 `6.8 벡터스토어 vs 데이터베이스 특징 비교`를 중심으로, 앞 장에서 사용한 FAISS, Chroma와 이후 DB 챗봇 실습에서 사용할 RDBMS 개념을 연결합니다.

## 이 장에서 다루는 내용

1. 벡터스토어가 필요한 이유
2. 벡터 검색과 일반 검색의 차이
3. 코사인 유사도와 유클리드 거리
4. Chroma
5. FAISS
6. MongoDB Atlas Vector Search
7. ElasticSearch
8. PostgreSQL + pgvector
9. MariaDB
10. RDBMS와 벡터 DB의 차이
11. 텍스트 검색, BM25, 벡터 검색
12. 저장소 선택 기준
13. 이 교재의 실습에서 FAISS와 Chroma를 쓰는 이유

RAG 시스템의 품질과 운영 방식은 어떤 벡터스토어를 선택하느냐에 크게 영향을 받습니다.


## 11.1 벡터스토어가 필요한 이유

RAG에서는 문서와 질문을 임베딩 벡터로 변환합니다.

문서가 몇 개뿐이라면 모든 벡터를 하나씩 비교해도 됩니다. 하지만 문서가 수천, 수만, 수백만 개가 되면 단순 비교는 느려집니다.

벡터스토어는 다음 역할을 합니다.

- 문서 청크의 임베딩 벡터 저장
- 원본 텍스트와 metadata 저장
- 질문 벡터와 유사한 문서 벡터 검색
- 대량 벡터 검색 최적화
- RAG Retriever의 기반 제공

즉 벡터스토어는 RAG 시스템의 검색 엔진입니다.


## 11.2 일반 검색과 벡터 검색의 차이

### 일반 키워드 검색

키워드 검색은 사용자가 입력한 단어가 문서에 있는지 찾습니다.

예시:

```text
질문: 고조선 건국
문서에 '고조선' 또는 '건국'이라는 단어가 있는지 검색
```

장점은 정확한 키워드 매칭에 강하다는 것입니다. 단점은 같은 의미를 다른 표현으로 쓴 경우 놓칠 수 있다는 것입니다.

### 벡터 검색

벡터 검색은 문장 의미를 벡터로 바꾼 뒤, 의미적으로 가까운 문서를 찾습니다.

예시:

```text
질문: 고조선은 언제 세워졌나요?
문서: 고조선은 기원전 2333년에 건국되었다고 전해진다.
```

질문과 문서가 정확히 같은 단어를 쓰지 않아도 의미가 비슷하면 검색될 수 있습니다.

RAG에서는 보통 벡터 검색을 기본으로 사용하고, 필요하면 키워드 검색과 결합합니다.


## 11.3 거리와 유사도

벡터 검색은 질문 벡터와 문서 벡터가 얼마나 가까운지 계산합니다.

대표 방식은 다음과 같습니다.

| 방식 | 설명 |
|---|---|
| 코사인 유사도 | 두 벡터의 방향이 얼마나 비슷한지 계산합니다. 텍스트 임베딩 검색에서 자주 사용합니다. |
| 유클리드 거리 | 두 벡터 사이의 직선 거리를 계산합니다. 거리가 작을수록 가깝습니다. |
| 내적 | 두 벡터의 곱을 통해 유사도를 계산합니다. |

PDF 교재의 벡터스토어 비교표에서도 코사인 유사도와 유클리드 거리 지원 여부가 반복해서 등장합니다.


## 11.4 Chroma

PDF 교재의 Chroma 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | 영속성 벡터 DB, 빠른 시작과 사용 편의성, SQLite 사용 |
| 특장점 | 개발 환경에 적합한 경량 솔루션 |
| 주요 지원 기능 | HNSW 기반 최근접 이웃 검색, 코사인 유사도 기본 지원 |

Chroma는 이 교재의 9장에서 사용했습니다.

사용 예:

```python
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_store"
)
```

Chroma가 적합한 경우:

- 빠르게 RAG 프로토타입을 만들 때
- 로컬에 벡터 DB를 저장하고 싶을 때
- 문서와 메타데이터를 함께 관리하고 싶을 때
- 서버 방식으로 중앙 벡터 DB를 운영해 보고 싶을 때


## 11.5 FAISS

PDF 교재의 FAISS 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | 고성능 벡터 검색 라이브러리, 독립형 DB가 아님, 보조 저장소 필요 |
| 특장점 | 매우 빠른 검색 속도와 대규모 데이터셋 처리에 최적화 |
| 주요 지원 기능 | 코사인, 유클리드 등 다양한 거리 함수 지원 |

FAISS는 이 교재의 8장과 10장에서 사용했습니다.

사용 예:

```python
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()
```

FAISS가 적합한 경우:

- 빠른 벡터 검색이 중요할 때
- 대량 벡터 검색 성능을 실험할 때
- 벡터 인덱스를 직접 관리할 수 있을 때
- 간단한 로컬 RAG 실습을 빠르게 구성할 때

주의: FAISS는 DB라기보다 검색 라이브러리이므로 영속 저장, 사용자 권한, 분산 운영, 메타데이터 관리 등은 별도로 고려해야 합니다.


## 11.6 MongoDB Atlas Vector Search

PDF 교재의 MongoDB 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | NoSQL DB, Atlas Vector Search 기능을 통해 벡터 검색 지원 |
| 특장점 | 기존 MongoDB 데이터와의 통합이 쉽고, 유연한 문서 기반 데이터 모델과 벡터 검색을 결합할 수 있음 |
| 주요 지원 기능 | 코사인, 유클리드 등 다양한 거리 함수 지원 |

MongoDB가 적합한 경우:

- 이미 MongoDB를 사용 중인 서비스
- JSON 문서 형태의 데이터를 저장하고 검색하는 서비스
- 일반 필터 조건과 벡터 검색을 함께 쓰고 싶은 경우
- 제품, 게시글, 문서, 사용자 로그 같은 문서형 데이터를 다루는 경우

예를 들어 상품 정보가 MongoDB에 저장되어 있다면, 상품 설명 임베딩을 함께 저장하고 의미 검색을 구현할 수 있습니다.


## 11.7 ElasticSearch

PDF 교재의 ElasticSearch 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | 전문적인 검색엔진, Dense Vector 필드를 통한 벡터 검색 지원, 텍스트 검색과 벡터 검색 결합이 쉬움 |
| 특장점 | TF-IDF, BM25 텍스트 검색과 벡터 검색을 강력하게 결합할 수 있고 복잡한 쿼리에 유리 |
| 주요 지원 기능 | 코사인 유사도, TF-IDF 기반 전문 텍스트 검색, BM25 포함 |

ElasticSearch가 적합한 경우:

- 기존에 검색 시스템을 운영 중인 서비스
- 키워드 검색과 의미 검색을 함께 써야 하는 경우
- 로그, 문서, 상품, 게시글 검색처럼 검색 품질이 중요한 경우
- 필터, 정렬, 집계, 하이라이팅 등 검색엔진 기능이 필요한 경우

RAG에서도 ElasticSearch는 hybrid search를 구현할 때 자주 고려됩니다.

Hybrid search는 키워드 검색과 벡터 검색을 함께 사용하는 방식입니다.


## 11.8 PostgreSQL + pgvector

PDF 교재의 PostgreSQL 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | `pgvector` 확장 기능을 통해 벡터 검색 지원, 정형 데이터와 통합 용이 |
| 특장점 | 정형 데이터, 트랜잭션 기능, 벡터 검색의 통합이 쉽고 안정성이 검증됨 |
| 주요 지원 기능 | HNSW 등 다양한 인덱스, 코사인/유클리드 등 다양한 거리 함수 지원 |

PostgreSQL + pgvector가 적합한 경우:

- 이미 PostgreSQL을 사용 중인 서비스
- 정형 데이터와 벡터 검색을 함께 써야 하는 경우
- 트랜잭션과 안정성이 중요한 경우
- 별도 벡터 DB를 추가하고 싶지 않은 경우

예를 들어 고객 테이블, 상품 테이블, 문서 테이블이 PostgreSQL에 있다면 임베딩 벡터 컬럼을 추가해 의미 검색을 붙일 수 있습니다.


## 11.9 MariaDB

PDF 교재의 MariaDB 설명은 다음과 같습니다.

| 구분 | 내용 |
|---|---|
| 주요 특징 | MyRocks 엔진의 인덱스 기능 활용 또는 외부 솔루션 연동 필요, 벡터 검색 기능은 상대적으로 약함 |
| 특장점 | 기존 RDBMS 인프라 활용 가능 |
| 주요 지원 기능 | 특화된 벡터 검색 알고리즘은 미지원, 외부 솔루션 연동 필요 |

MariaDB가 적합한 경우:

- 이미 MariaDB 기반 시스템을 운영 중인 경우
- 정형 데이터 조회가 중심인 경우
- 벡터 검색은 FAISS, Chroma, ElasticSearch 등 외부 솔루션과 연동할 수 있는 경우

이 교재의 `llm2.ipynb`에는 MySQL 계열 DB에 접속해 자연어 질문을 SQL로 바꾸는 DB 챗봇 예제가 있습니다. 이 예제는 벡터 검색보다는 RDBMS의 정형 쿼리와 LLM을 결합하는 방식입니다.


## 11.10 RDBMS와 벡터스토어의 차이

RDBMS와 벡터스토어는 목적이 다릅니다.

| 구분 | RDBMS | 벡터스토어 |
|---|---|---|
| 데이터 형태 | 행과 열로 구성된 정형 데이터 | 고차원 벡터와 원문 텍스트 |
| 검색 방식 | SQL 조건 검색, 조인, 집계 | 벡터 유사도 검색 |
| 강점 | 트랜잭션, 정합성, 정형 질의 | 의미 기반 검색, 유사 문서 검색 |
| 예시 | MySQL, MariaDB, PostgreSQL | FAISS, Chroma, Pinecone, Weaviate |
| LLM 활용 | 자연어를 SQL로 변환, 결과 설명 | RAG 문서 검색 |

예를 들어 `2024년 매출이 가장 높은 고객은?` 같은 질문은 RDBMS SQL이 적합합니다.

반면 `환불 규정이 어떻게 돼?`처럼 문서 의미를 찾아야 하는 질문은 벡터스토어 RAG가 적합합니다.


## 11.11 텍스트 검색과 BM25

PDF의 ElasticSearch 설명에는 BM25가 등장합니다.

BM25는 전통적인 검색엔진에서 많이 사용하는 키워드 기반 랭킹 알고리즘입니다.

BM25는 다음 요소를 고려합니다.

- 검색어가 문서에 얼마나 자주 등장하는지
- 검색어가 전체 문서에서 얼마나 희귀한지
- 문서 길이가 얼마나 긴지

BM25는 정확한 키워드 검색에 강합니다. 반면 벡터 검색은 의미적으로 비슷한 문장을 찾는 데 강합니다.

그래서 실무 RAG에서는 다음처럼 둘을 결합하기도 합니다.

```text
Hybrid Search = Keyword Search(BM25) + Vector Search
```

예를 들어 제품 코드, 법 조항 번호, 사람 이름처럼 정확한 키워드가 중요한 경우 BM25가 도움이 됩니다.


## 11.12 저장소 선택 기준

벡터스토어나 DB를 선택할 때는 다음 기준을 봅니다.

| 기준 | 고려할 질문 |
|---|---|
| 데이터 규모 | 문서가 수백 개인가, 수백만 개인가? |
| 운영 환경 | 로컬 실습인가, 상용 서비스인가? |
| 영속성 | 재시작 후에도 벡터가 유지되어야 하는가? |
| 검색 품질 | 키워드 검색도 필요한가, 의미 검색만 필요한가? |
| 기존 인프라 | 이미 MongoDB, PostgreSQL, ElasticSearch를 쓰고 있는가? |
| 보안 | 사내망 또는 폐쇄망이 필요한가? |
| 메타데이터 | 출처, 날짜, 권한, 카테고리 필터가 필요한가? |
| 동시 사용자 | 여러 사용자가 동시에 검색하는가? |
| 확장성 | 서버 분리, 클러스터, 백업이 필요한가? |

학습과 프로토타입은 FAISS나 Chroma로 시작하고, 운영 요구사항이 커지면 MongoDB, ElasticSearch, PostgreSQL pgvector 같은 선택지를 고려할 수 있습니다.


## 11.13 실습별 저장소 선택 이유

이 교재의 실습에서 사용하는 저장소와 이유는 다음과 같습니다.

| 실습 | 저장소 | 이유 |
|---|---|---|
| 간단한 LangChain 검색 예제 | FAISS | 짧은 텍스트로 벡터 검색 흐름을 빠르게 확인 |
| history.txt RAG | FAISS | PDF 6.6 RAG 예제와 유사한 기본 구조 학습 |
| CSV RAG 챗봇 | FAISS | DataFrame을 텍스트로 바꿔 간단히 검색 실습 |
| URL RAG | FAISS | URL마다 일회성 벡터스토어 생성이 쉬움 |
| 로컬 Chroma RAG | Chroma | 벡터 저장소를 폴더에 영속 저장하는 흐름 학습 |
| Chroma 서버 RAG | Chroma | 중앙 벡터 DB와 로컬 LLM 분리 구조 학습 |
| DB 질의 챗봇 | MySQL/MariaDB 계열 | 정형 데이터에 SQL 질의 생성과 실행 |

즉 FAISS는 가볍고 빠른 실습용, Chroma는 저장과 서버 구조 실습용, RDBMS는 정형 데이터 질의 실습용으로 사용됩니다.


## 11.14 벡터스토어와 권한 관리

실무 RAG에서는 모든 문서를 모든 사용자가 검색하면 안 되는 경우가 많습니다.

예시:

- 부서별 내부 문서
- 고객별 계약서
- 관리자 전용 매뉴얼
- 유료 사용자 전용 콘텐츠

이 경우 metadata 필터링이 중요합니다.

예를 들어 문서 청크에 다음 metadata를 저장할 수 있습니다.

```json
{
  "source": "hr_policy.pdf",
  "department": "HR",
  "access_level": "internal",
  "created_at": "2026-07-01"
}
```

검색 시 사용자의 권한에 맞는 문서만 검색하도록 필터링해야 합니다.


## 11.15 벡터스토어 운영 시 체크리스트

- [ ] 문서 출처 metadata를 저장하는가?
- [ ] 문서 버전과 업데이트 날짜를 관리하는가?
- [ ] 삭제된 문서의 벡터도 삭제되는가?
- [ ] 사용자의 권한에 맞게 검색 결과를 제한하는가?
- [ ] 임베딩 모델 변경 시 기존 벡터 재생성 계획이 있는가?
- [ ] 검색 품질을 평가할 테스트 질문 세트가 있는가?
- [ ] 검색 결과와 최종 답변을 로그로 남기는가?
- [ ] 개인정보나 민감정보가 벡터스토어에 저장되는지 검토했는가?
- [ ] 백업과 복구 전략이 있는가?
- [ ] 운영 환경에서 동시 요청을 견딜 수 있는가?


## 11.16 자주 발생하는 선택 실수

### 1. 무조건 벡터 DB만 사용

정확한 숫자 계산이나 조건 검색은 SQL이 더 적합할 수 있습니다.

예: `2025년 3월 매출 합계는?`

### 2. 무조건 RDBMS만 사용

문서 의미 검색은 SQL LIKE 검색만으로는 부족할 수 있습니다.

예: `휴가 규정 중 병가와 관련된 내용 알려줘`

### 3. 임베딩 모델을 중간에 바꾸고 기존 벡터를 그대로 사용

임베딩 모델이 바뀌면 벡터 공간도 달라집니다. 기존 문서를 새 모델로 다시 임베딩해야 합니다.

### 4. 출처 metadata를 저장하지 않음

나중에 답변 근거를 표시하거나 문서 업데이트를 관리하기 어렵습니다.

### 5. 청크 전략을 고려하지 않음

청크 크기와 overlap이 검색 품질에 큰 영향을 줍니다.


## 11.17 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- 벡터스토어는 RAG에서 문서 임베딩을 저장하고 유사 문서를 검색하는 핵심 요소입니다.
- 키워드 검색은 정확한 단어 매칭에 강하고, 벡터 검색은 의미 검색에 강합니다.
- Chroma는 빠르게 시작하기 좋은 영속성 벡터 DB입니다.
- FAISS는 고성능 벡터 검색 라이브러리입니다.
- MongoDB는 문서형 데이터와 벡터 검색을 결합하기 좋습니다.
- ElasticSearch는 BM25 기반 텍스트 검색과 벡터 검색을 결합하기 좋습니다.
- PostgreSQL pgvector는 정형 데이터와 벡터 검색을 함께 쓰기 좋습니다.
- MariaDB는 기존 RDBMS 인프라 활용에는 좋지만 벡터 검색 특화 기능은 상대적으로 약합니다.
- RDBMS는 정형 질의와 트랜잭션에 강하고, 벡터스토어는 의미 검색에 강합니다.
- 실습에서는 FAISS로 기본 RAG를 익히고, Chroma로 영속 저장과 서버 구조를 익힙니다.

다음 장에서는 자연어 질문을 SQL로 변환하고 DB 결과를 답변하는 DB 접속 LLM 챗봇을 제작합니다.
